In [31]:
import sys
import glob
import pickle
import numpy as np
import pandas as pd
from functools import reduce

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")

import Chronocell

In [32]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.5_208_genes_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [83]:
t = traj.t

In [58]:
# Y = traj.X
Q = traj.Q[:, 0, :] 
# tau = traj.tau # State transition times (global)
#  = traj.t
# theta = traj.theta
# topo = traj.topo
# state_grid = np.searchsorted(tau, t, side="left") - 1
# state_grid[0] = 0 

## Get genes of interest

In [3]:
genes = pd.read_csv("eLNPs_var>1.5_208_genes.csv")
shared_genes = pd.read_csv("RNA_vs_ADT_corr_meanExpr.csv")
gene_idx = genes['Gene_symbol'].isin(shared_genes['Gene']).tolist()

In [4]:
genes.iloc[gene_idx]['Gene_symbol'].tolist()

['Cxcr4',
 'Icos',
 'Cd48',
 'Sbno2',
 'Cd24a',
 'Cd63',
 'Cd68',
 'Ccr7',
 'Havcr2',
 'Ighm',
 'Cd83',
 'Lgals3',
 'Il7r',
 'Ly6a',
 'Ly6e',
 'Cd86',
 'Cd200',
 'Cd80',
 'Btla',
 'Cd274',
 'Fas',
 'Dpp4',
 'Il2ra',
 'Itga4',
 'Procr',
 'Cd1d1',
 'Vcam1',
 'Tnfrsf4',
 'Tnfrsf18',
 'Cd36',
 'Sbno1',
 'Flt3',
 'Cd38',
 'Kit',
 'Cd9',
 'Cd69',
 'Klrk1',
 'Cd8a',
 'Clec12a',
 'Clec9a',
 'Pirb',
 'Ilk',
 'Il4ra',
 'Itgal',
 'Itgax',
 'Bst2',
 'Lamp1',
 'Itgb1',
 'Xcr1',
 'Icam1',
 'Jaml']

## Get half-life data

In [5]:
data_dir = "/mnt/lareaulab/reliscu/projects/Chronocell/data/experimental_parameters/protein_degradation_rates"
file_list = glob.glob(f"{data_dir}/*standardized.csv")
subset_columns = ['Gene', 'Half-life']

df_list = []
for file in [file for file in file_list if "Mouse" in file]:
    df = pd.read_csv(file)
    study = df['Study'].iloc[0]
    cell_type = df['Cell_type'].iloc[0]
    df = df[subset_columns]
    df = df.rename(columns={"Half-life": f"Half-life_{study}_{cell_type}", 
                            "Cell_type": f"Cell_type_{study}"}) 
    df_list.append(df)

In [6]:
df_list[0].head()

,Gene,Half-life_McShane2016_NIH3T3 fibroblasts
0,Cul4b,31.378265
1,Dhx8,28.700000
2,Znf335,3.599724
3,Arfgef2,24.371302
4,Cdc27,21.057713


In [7]:
df = reduce(lambda x, y: pd.merge(x, y, on="Gene", how="outer"), df_list)


## Subset half-life data to genes of interest

In [8]:
df_subset = df.loc[df['Gene'].isin(genes.iloc[gene_idx]['Gene_symbol'])]

## Convert to degradation rate

In [9]:
half_life_cols = [col for col in df_subset.columns if "Half-life" in col]
df_deg = np.log(2)/df_subset[half_life_cols]
median_deg = np.nanmedian(df_deg, axis=1)

In [10]:
df_subset['Gene']

1962       Bst2
2973       Cd63
2976        Cd9
7581        Ilk
7683      Itgb1
8793      Lamp1
8875     Lgals3
12968     Procr
16288     Sbno1
16289     Sbno2
18975     Vcam1
Name: Gene, dtype: object

## Impute protein

In [ ]:
Q_max_idx = np.argmax(Q, axis=1) # Each cell's time along trajectory

i = 0 # Gene index

#######

gene = df_subset['Gene'].iloc[0]
file_path = f"/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint/data/RNA_history_per_gene/eLNPs_var>1.5_208_genes_{gene}_S_RNA_history.pkl"

with open(file_path, "rb") as f:
    X_bw = pickle.load(f)
    
deg_rate = median_deg[0]

y0 = X_bw[:, 0] # Steady-state RNA abundance per gene
ss_rate = deg_rate # Steady-state protein production rate = transl_rate/deg_rate
p0 = ss_rate * y0 # Initial protein abundance assuming steady-state

p_old = p0 * np.exp(-deg_rate * t[Q_max_idx]) # Pre-existing protein that has not yet degraded at the time each cell was observed

t_reshape = t.reshape((-1, 1))
t_diff = t_reshape.T - t_reshape
t_diff = t_diff[:, Q_max_idx] # Each column corresponds to the time steps leading up to the observed time for a given cell

decay_matrix = np.exp(-t_diff * deg_rate) # Decay_matrix[m, i] = decay factor for protein abundance at t_m from RNA available at t_i 
mask = (t_diff >= 0)
mask = np.broadcast_to(mask, decay_matrix.shape)
decay_matrix = np.where(mask, decay_matrix, 0) # Protein abundance at time t_m can't come from RNA at time t_i > t_m

dt = np.diff(t, prepend=t[1])
y_dt = X_bw * dt # Multiply each timepoint's RNA by its corresponding time step size (Riemann approximation) 
y_dt[y_dt < 0] = 0 # X_bw was populated with '-1' for time points later than the cell was observed
# Note to self: protein produced from RNA from time t_m can only be produced during that interval, hence: scale RNA contribution at that time step by its time step size

p_new = (decay_matrix.T * y_dt).sum(axis=1) # Integrate RNA counts still surviving up to each time point (until the cell was observed)
P = p_old + p_new # Protein abundance in each cell = pre-existing protein + newly synthesized protein
    

In [197]:
P

array([ 2539.83864089,  2539.83372264,  9674.4824337 , ...,
       13895.76460394,  2531.85261389, 10190.38604085])

In [135]:
decay_matrix * y_dt

array([[0.        , 0.08030055, 0.0797962 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.08080808, 0.08030055, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.08080808, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [132]:
p_new

array([0.16009674, 0.16110863, 0.08080808, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [131]:
P

array([0.2226361 , 0.22364798, 0.14334744, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253936,
       0.06253936, 0.06253936, 0.06253936, 0.06253936, 0.06253

In [117]:
P

array([0.23806606, 0.23858823, 0.15780106, 0.0765094 , 0.07602887,
       0.07555135, 0.07507683, 0.07460529, 0.07413672, 0.07367108,
       0.07320837, 0.07274857, 0.07229166, 0.07183761, 0.07138642,
       0.07093806, 0.07049251, 0.07004977, 0.0696098 , 0.0691726 ,
       0.06873815, 0.06830642, 0.0678774 , 0.06745108, 0.06702744,
       0.06660646, 0.06618812, 0.06577241, 0.06535931, 0.06494881,
       0.06454088, 0.06413551, 0.06373269, 0.06333241, 0.06293463,
       0.06253936, 0.06214656, 0.06175624, 0.06136836, 0.06098292,
       0.0605999 , 0.06021929, 0.05984107, 0.05946522, 0.05909174,
       0.0587206 , 0.05835179, 0.05798529, 0.0576211 , 0.0572592 ,
       0.05689957, 0.0565422 , 0.05618707, 0.05583417, 0.05548349,
       0.05513502, 0.05478873, 0.05444461, 0.05410266, 0.05376286,
       0.05342519, 0.05308964, 0.05275619, 0.05242485, 0.05209558,
       0.05176838, 0.05144324, 0.05112013, 0.05079906, 0.05048001,
       0.05016295, 0.04984789, 0.04953481, 0.0492237 , 0.04891

In [ ]:
xp(-deg_rate * t) # Pre-existing protein that has not yet degraded

t_diff = t_l - t_l.T # Rows = target time; columns = past times; e.g. t_diff[m, i] = time difference between t_m and t_i 
decay_matrix = np.exp(-t_diff[:, :, None] * deg_rates) # Decay_matrix[m, i, p] = decay factor for protein abundance at t_m from RNA available at t_i for gene p
mask = (t_diff >= 0)[:, :, None]
mask = np.broadcast_to(mask, decay_matrix.shape)
decay_matrix = np.where(mask, decay_matrix, 0) # Protein abundance at time t_m can't come from RNA at time t_i > t_m

y_l_dt = y_l * dt[:, None] # Multiply each timepoint's RNA by its corresponding time step size (Riemann approximation)
# Note to self: protein produced from RNA from time t = m can only be produced during the interval it was measured over, hence: scale RNA contribution at that time step by its time step size
protein_contrib = (decay_matrix * y_l_dt[None, :, :]).sum(axis=1) # Integrate RNA counts still surviving up to each time point
# Note to self: protein_contrib[target_idx, gene_idx] = np.sum(decay_matrix[target_time, :(target_time + 1), gene_idx] * y_l_dt[None, :(target_time + 1), gene_idx])
P[l*n:(l+1)*n] = p_l + transl_rates * protein_contrib # Protein abundance in each cell = pre-existing protein + newly synthesized protein
    

In [ ]:
def simulate_protein_from_RNA(Y, topo, true_t, true_l, phi, random_seed=0):
    ## phi: Protein params
    np.random.seed(random_seed)
    
    L = len(topo) # No. lineages
    n = Y.shape[0] // L # No. cells per lineage
    p = Y.shape[1] # No. genes
    
    y0 = Y[0, :, 1] # RNA abundance per gene at state 0
    ss_rate = phi[:, 0] / phi[:, -1] # Steady-state protein production rate = transl_rate/deg_rate
    p0 = ss_rate * y0 # Initial protein abundance assuming steady-state
    
    # Protein production paramters:
    transl_rates = phi[:, 0].T
    deg_rates = phi[:, -1].reshape((1, -1))
    
    P = np.zeros((n*L, p))
    
    for l in range(L):
        t_l = true_t[true_l == l] # Time points/cells in lineage l     
        dt = np.diff(t_l, prepend=t_l[0]) # Time step size for each cell along the trajectory
        t_l = t_l.reshape((-1, 1)) 
        y_l = Y[l*n:(l+1)*n, :, 1] # Spliced RNAs for lineage l
        
        p_l = p0 * np.exp(-deg_rates * t_l) # Pre-existing protein that has not yet degraded

        t_diff = t_l - t_l.T # Rows = target time; columns = past times; e.g. t_diff[m, i] = time difference between t_m and t_i 
        decay_matrix = np.exp(-t_diff[:, :, None] * deg_rates) # Decay_matrix[m, i, p] = decay factor for protein abundance at t_m from RNA available at t_i for gene p
        mask = (t_diff >= 0)[:, :, None]
        mask = np.broadcast_to(mask, decay_matrix.shape)
        decay_matrix = np.where(mask, decay_matrix, 0) # Protein abundance at time t_m can't come from RNA at time t_i > t_m
        
        y_l_dt = y_l * dt[:, None] # Multiply each timepoint's RNA by its corresponding time step size (Riemann approximation)
        # Note to self: protein produced from RNA from time t = m can only be produced during the interval it was measured over, hence: scale RNA contribution at that time step by its time step size
        protein_contrib = (decay_matrix * y_l_dt[None, :, :]).sum(axis=1) # Integrate RNA counts still surviving up to each time point
        # Note to self: protein_contrib[target_idx, gene_idx] = np.sum(decay_matrix[target_time, :(target_time + 1), gene_idx] * y_l_dt[None, :(target_time + 1), gene_idx])
        P[l*n:(l+1)*n] = p_l + transl_rates * protein_contrib # Protein abundance in each cell = pre-existing protein + newly synthesized protein
            
    return P